In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.append('../')
from src.neural_net import NeuralNetworkScratch
print("Import successful")

Import successful


In [2]:
np.random.seed(42)
X_small = np.random.randn(3, 4)       # 3 samples, 4 features (tiny test)
y_small = np.array([[1.0],[2.0],[3.0]])

nn_test = NeuralNetworkScratch(input_size=4)
y_pred, cache = nn_test.forward(X_small)
grads = nn_test.backward(cache, y_small)

print("Gradient shapes:")
for key, val in grads.items():
    print(f"  {key}: {val.shape}")

print("\nAny NaN in gradients?", any(np.any(np.isnan(g)) for g in grads.values()))
print("Any Inf in gradients?", any(np.any(np.isinf(g)) for g in grads.values()))

Gradient shapes:
  dW1: (4, 64)
  db1: (1, 64)
  dW2: (64, 32)
  db2: (1, 32)
  dW3: (32, 1)
  db3: (1, 1)

Any NaN in gradients? False
Any Inf in gradients? False


In [3]:
df = pd.read_csv('data/final_prepared.csv')
X = df.drop('median_house_value', axis=1).values
y = df['median_house_value'].values.reshape(-1, 1)

nn_real = NeuralNetworkScratch(input_size=X.shape[1])
y_pred_real, cache_real = nn_real.forward(X)
grads_real = nn_real.backward(cache_real, y)

print("Gradient shapes on real data:")
for key, val in grads_real.items():
    print(f"  {key}: {val.shape}")

Gradient shapes on real data:
  dW1: (12, 64)
  db1: (1, 64)
  dW2: (64, 32)
  db2: (1, 32)
  dW3: (32, 1)
  db3: (1, 1)


In [4]:
lr = 0.001
mse_before = np.mean((y_pred_real - y) ** 2)

params = nn_real.get_params()
new_params = {
    'W1': params['W1'] - lr * grads_real['dW1'],
    'b1': params['b1'] - lr * grads_real['db1'],
    'W2': params['W2'] - lr * grads_real['dW2'],
    'b2': params['b2'] - lr * grads_real['db2'],
    'W3': params['W3'] - lr * grads_real['dW3'],
    'b3': params['b3'] - lr * grads_real['db3'],
}
nn_real.set_params(new_params)

y_pred_after, _ = nn_real.forward(X)
mse_after = np.mean((y_pred_after - y) ** 2)

print(f"MSE before update: {mse_before:.2f}")
print(f"MSE after  update: {mse_after:.2f}")
print("Loss went DOWN:", mse_after < mse_before)

MSE before update: 50275768757.10
MSE after  update: 50114064021.87
Loss went DOWN: True


In [5]:
params = nn_real.get_params()

param_to_grad = {
    'W1':'dW1', 'b1':'db1',
    'W2':'dW2', 'b2':'db2',
    'W3':'dW3', 'b3':'db3'
}

print("Optimizer interface alignment check:")
all_ok = True
for p_key, g_key in param_to_grad.items():
    p_shape = params[p_key].shape
    g_shape = grads_real[g_key].shape
    match = (p_shape == g_shape)
    print(f"  {p_key}{p_shape} <-> {g_key}{g_shape} : {'OK' if match else 'MISMATCH'}")
    if not match:
        all_ok = False

print()
print("Member 3 optimizer interface:", "READY" if all_ok else "FIX MISMATCHES ABOVE")


Optimizer interface alignment check:
  W1(12, 64) <-> dW1(12, 64) : OK
  b1(1, 64) <-> db1(1, 64) : OK
  W2(64, 32) <-> dW2(64, 32) : OK
  b2(1, 32) <-> db2(1, 32) : OK
  W3(32, 1) <-> dW3(32, 1) : OK
  b3(1, 1) <-> db3(1, 1) : OK

Member 3 optimizer interface: READY
